In [14]:
!pip install -q --upgrade \
    pdfplumber \
    tqdm

# Đại Việt Sử Ký Toàn Thư

In [15]:
import re
import json
import pdfplumber
from tqdm import tqdm

PDF_PATH      = "DVSK_NhaTran.pdf"
OUTPUT_JSON   = "DVSKTT_Chunks.json"
BOOK_NAME     = "Đại Việt Sử Ký Toàn Thư"

In [16]:

# Separator ưu tiên tách tại ranh giới đoạn/câu tự nhiên của văn bản sử ký
SEPARATORS = [
    "\n\n",   # Ranh giới đoạn (ưu tiên cao nhất)
    "\n",     # Ranh giới dòng
    ". ",     # Cuối câu
    "! ", "? ", "; ", ", ", " ", "",
]

PAGE_MARKER_FMT = "\n[PAGE:{page}]\n"

HEADER_FOOTER_PATTERNS = [
    re.compile(r'^\s*\d{1,4}\s+Đại Việt Sử Ký Toàn Thư.*$', re.MULTILINE | re.IGNORECASE),
    re.compile(r'^Đại Việt Sử Ký Toàn Thư.*\d{1,4}\s*$',    re.MULTILINE | re.IGNORECASE),
    re.compile(r'^\s*\d{1,4}\s*$', re.MULTILINE),
    re.compile(r'^\s*(Đại Việt Sử Ký Toàn Thư|Bản Kỷ|Ngoại Kỷ|Quyển\s+[IVXLC]+)\s*$',
               re.MULTILINE | re.IGNORECASE),
]

BOOK_PAGE_RE = re.compile(
    r'^\s*(\d{1,4})\s+Đại Việt Sử Ký Toàn Thư',
    re.MULTILINE | re.IGNORECASE
)

FOOTNOTE_LINE_RE = re.compile(
    r'^\s{0,9}(\d{1,2})(?:\.|\)|\]|\s{1,4}[A-ZÀ-Ỹ\[])'
)

def replace_superscripts(text: str) -> str:
    text = re.sub(r'([a-zA-ZÀ-ỹ\)])(\d+)(?=\s|[,.\!\?;:\)\]\n]|$)', r'\1 [\2]', text)
    return text


# header and footer
def remove_headers(text: str) -> str:
    for p in HEADER_FOOTER_PATTERNS:
        text = p.sub('', text)
    return text


def extract_book_page_number(text: str):

    m = BOOK_PAGE_RE.search(text)

    if m:
        return int(m.group(1))

    return None

# tách footer (tách phần chú thích)

def extract_footnotes_as_dict(text: str) -> tuple[str, dict]:
    """Tách footnote từ boundary hoặc từ nội dung"""
    if "---FOOTNOTE_BOUNDARY---" in text:
        main_part, foot_part = text.split("---FOOTNOTE_BOUNDARY---", 1)
    else:
        main_part = text
        foot_part = ""

    footnote_dict = {}

    # Tìm tất cả footnote trong foot_part trước
    if foot_part.strip():
        matches = list(FOOTNOTE_LINE_RE.finditer(foot_part))
        for i, m in enumerate(matches):
            num = m.group(1)
            start = m.end()
            end = matches[i+1].start() if i+1 < len(matches) else len(foot_part)
            content = foot_part[start:end].strip()
            footnote_dict[num] = re.sub(r'\s+', ' ', content)

    # Nếu không có boundary thì thử quét toàn bộ (backup)
    if not footnote_dict:
        main_part, footnote_dict = _extract_footnotes_fallback(main_part)

    return main_part.strip(), footnote_dict


def _extract_footnotes_fallback(text: str) -> tuple[str, dict]:
    """Fallback khi không có boundary"""
    lines = text.split('\n')
    footnote_dict = {}
    main_lines = []
    current_key = None
    current_parts = []

    for line in lines:
        m = FOOTNOTE_LINE_RE.match(line)
        if m:
            if current_key:
                footnote_dict[current_key] = ' '.join(p.strip() for p in current_parts if p.strip())

            current_key = m.group(1)
            content = re.sub(r'^\s*\d{1,3}\s*[\.\)\]]\s*', '', line).strip()
            current_parts = [content]
        elif current_key and line.strip():
            current_parts.append(line)
        else:
            if current_key is None:
                main_lines.append(line)

    if current_key:
        footnote_dict[current_key] = ' '.join(p.strip() for p in current_parts if p.strip())

    return '\n'.join(main_lines), footnote_dict

# Làm sách trang
def normalize_whitespace(text: str) -> str:
    text = re.sub(r'[ \t]+\n', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(
        r'(?<![.!?;:])\n(?!\n)',
        ' ',
        text
    )
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def clean_page(raw_text: str) -> tuple[str, dict]:
    if not raw_text:
        return "", {}
    text = replace_superscripts(raw_text)   # abc12 -> abc [12] || abc)12 -> abc) [12]
    text = re.sub(r'\[\d{1,2}[ab]\]', '', text)
    text = remove_headers(text)             # xóa header
    text, fn_dict = extract_footnotes_as_dict(text)
    text = normalize_whitespace(text)
    return text, fn_dict

def detect_footer_separator(page) -> float | None:
    """Tìm đường kẻ ngang ở nửa dưới trang một cách chính xác hơn"""
    page_height = page.height
    page_width = page.width

    # Thu thập tất cả lines và rects
    objects = page.lines + page.rects + page.curves

    candidates = []
    for obj in objects:
        x0, y0, x1, y1 = obj.get('x0', 0), obj.get('y0', 0), obj.get('x1', 0), obj.get('y1', 0)
        width = abs(x1 - x0)
        height = abs(y1 - y0)

        # Điều kiện đường kẻ ngang:
        if (height < 3 and                     # rất mỏng
            width > page_width * 0.6 and       # dài ít nhất 60% trang
            y0 > page_height * 0.55):          # nằm ở nửa dưới
            candidates.append(y0)

    if candidates:
        # Lấy đường kẻ cao nhất (gần cuối trang nhất)
        return max(candidates)
    return None


def extract_all_pages(pdf_path: str):
    # START_PAGE = 1
    # END_PAGE   = 10
    results = []

    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(tqdm(pdf.pages, desc=" Đang xử lý trang"), start=1):
            # if i < START_PAGE:
            #     continue
            # if i > END_PAGE:
            #     break
            hehe = page.extract_text(x_tolerance=3, y_tolerance=3)
            book_page = extract_book_page_number(hehe)

            separator_y = detect_footer_separator(page)

            if separator_y:
                # === CÓ ĐƯỜNG KẼ → Tách rõ ràng main + footer ===
                main_area = page.crop((0, 0, page.width, separator_y - 2))      # trừ margin nhỏ
                foot_area = page.crop((0, separator_y + 2, page.width, page.height))

                raw_main = main_area.extract_text(x_tolerance=3, y_tolerance=3) or ""
                raw_foot = foot_area.extract_text(x_tolerance=3, y_tolerance=3) or ""

                raw_text = raw_main.strip() + "\n---FOOTNOTE_BOUNDARY---\n" + raw_foot.strip()
            else:
                # === Không tìm thấy đường kẻ → lấy toàn bộ ===
                raw_text = page.extract_text(x_tolerance=3, y_tolerance=3) or ""

            if not raw_text.strip():
                continue

            text, fn_dict = clean_page(raw_text)

            results.append({
                "page_num": book_page,
                "text": text,
                "footnotes": fn_dict
            })

    return results



# Ghép TẤT CẢ trang thành một chuỗi lớn.
# Nhúng page marker [[[PAGE:n]]] vào giữa để sau split vẫn biết chunk ở trang nào.

# Returns:
#     (full_text, page_footnotes_map)
#     page_footnotes_map: {page_num: footnote_dict}

def build_full_text(pages: list[dict]) -> tuple[str, dict]:

    parts = []
    page_fn_map = {}

    for page in pages:
        pn = page["page_num"]
        page_fn_map[pn] = page["footnotes"]
        # Đặt marker trước mỗi trang để track page boundary sau split
        parts.append(PAGE_MARKER_FMT.format(page=pn))
        parts.append(page["text"])

    return "".join(parts), page_fn_map


In [17]:

def main():
    print("  XỬ LÝ PDF - ĐẠI VIỆT SỬ KÝ TOÀN THƯ ")

# chuyển từ pdf -> json. Tải về 2 file.
# 1 file chứa các trang đơn lẻ.
# 1 file ghép toàn bộ trang lại với nhau.
    pages = extract_all_pages(PDF_PATH)

    with open("pages_DVSK_data.json", "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)

    print("Đã lưu pages_data.json")



    full_text, page_fn_map = build_full_text(pages)

    full_text_data = {
        "book_name": BOOK_NAME,
        "full_text": full_text,
        "page_footnotes_map": page_fn_map
    }

    with open("full_DVSK_data.json", "w", encoding="utf-8") as f:
        json.dump(full_text_data, f, ensure_ascii=False, indent=2)

    print(" Đã lưu full_text_data.json")

    from google.colab import files

    files.download("pages_DVSK_data.json")
    files.download("full_DVSK_data.json")


if __name__ == "__main__":
    main()


  XỬ LÝ PDF - ĐẠI VIỆT SỬ KÝ TOÀN THƯ 


 Đang xử lý trang: 100%|██████████| 154/154 [01:40<00:00,  1.53it/s]

Đã lưu pages_data.json
 Đã lưu full_text_data.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Khâm Định Việt Sử Thông Giám Cương Mục

In [18]:

import re
import json
import pdfplumber
from tqdm import tqdm

PDF_PATH      = "KDVSTGCMQSQTN.pdf"
OUTPUT_JSON   = "KDVSTGCMQSQTN_chunks.json"
BOOK_NAME     = "Khâm Định Việt Sử Thông Giám Cương Mục"

HEADER_FOOTER_PATTERNS = [
    re.compile(r'^\s*\d{1,4}\s+Khâm Định Việt Sử Thông Giám Cương Mục.*$', re.MULTILINE | re.IGNORECASE),
    re.compile(r'^Khâm Định Việt Sử Thông Giám Cương Mục.*\d{1,4}\s*$',    re.MULTILINE | re.IGNORECASE),
    re.compile(r'^\s*\d{1,4}\s*$', re.MULTILINE),
    re.compile(r'^\s*(Khâm Định Việt Sử Thông Giám Cương Mục|Bản Kỷ|Ngoại Kỷ|Quyển\s+[IVXLC]+)\s*$',
               re.MULTILINE | re.IGNORECASE),
]


BOOK_PAGE_RE = re.compile(
    r'^\s*(\d{1,4})\s+Khâm Định Việt Sử Thông Giám Cương Mục.',
    re.MULTILINE | re.IGNORECASE
)


In [19]:
# @title

def main():

    print("  XỬ LÝ PDF - Khâm Định Việt Sử Thông Giám Cương Mục")

    pages = extract_all_pages(PDF_PATH)

    with open("pages_KhamDinh_data.json", "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)

    full_text, page_fn_map = build_full_text(pages)

    full_text_data = {
        "book_name": BOOK_NAME,
        "full_text": full_text,
        "page_footnotes_map": page_fn_map
    }

    with open("full_KhamDinh_data.json", "w", encoding="utf-8") as f:
        json.dump(full_text_data, f, ensure_ascii=False, indent=2)


    from google.colab import files

    files.download("pages_KhamDinh_data.json")
    files.download("full_KhamDinh_data.json")


if __name__ == "__main__":
    main()

  XỬ LÝ PDF - Khâm Định Việt Sử Thông Giám Cương Mục


 Đang xử lý trang: 100%|██████████| 165/165 [01:38<00:00,  1.68it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Việt Sử Toàn Thư


In [20]:

import re
import json
import pdfplumber
from tqdm import tqdm

PDF_PATH      = "VSTT.pdf"
OUTPUT_JSON   = "VSTT_chunks.json"
BOOK_NAME     = "Việt Sử Toàn Thư"


HEADER_FOOTER_PATTERNS = [
    re.compile(r'^\s*\d{1,4}\s+Việt Sử Toàn Thư.*$', re.MULTILINE | re.IGNORECASE),
    re.compile(r'^Việt Sử Toàn Thư.*\d{1,4}\s*$',    re.MULTILINE | re.IGNORECASE),
    re.compile(r'^\s*\d{1,4}\s*$', re.MULTILINE),
    re.compile(r'^\s*(Việt Sử Toàn Thư|Bản Kỷ|Ngoại Kỷ|Quyển\s+[IVXLC]+)\s*$',
               re.MULTILINE | re.IGNORECASE),
]


BOOK_PAGE_RE = re.compile(
    r'^\s*(\d{1,4})\s+Việt Sử Toàn Thư',
    re.MULTILINE | re.IGNORECASE
)




In [21]:

def main():
    print(f"Sách: {BOOK_NAME}")
    pages = extract_all_pages(PDF_PATH)

    with open("pages_VietSu_data.json", "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)

    full_text, page_fn_map = build_full_text(pages)

    full_text_data = {
        "book_name": BOOK_NAME,
        "full_text": full_text,
        "page_footnotes_map": page_fn_map
    }

    with open("full_VietSu_data.json", "w", encoding="utf-8") as f:
        json.dump(full_text_data, f, ensure_ascii=False, indent=2)


    from google.colab import files

    files.download("pages_VietSu_data.json")
    files.download("full_VietSu_data.json")


if __name__ == "__main__":
    main()

Sách: Việt Sử Toàn Thư


 Đang xử lý trang: 100%|██████████| 89/89 [00:42<00:00,  2.08it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Vương Triều Trần (1226-1400) - Vũ Văn Quân

In [22]:
import re
import json
from tqdm import tqdm
import pdfplumber

PDF_PATH      = "VTT.pdf"
OUTPUT_JSON   = "VTT_chunks.json"
BOOK_NAME     = "Vương Triều Trần (1226-1400)"

# ====================== PATTERNS ======================
HEADER_FOOTER_PATTERNS = [
    re.compile(r'^\s*[A-ZÀ-Ỹ\s]+\s*\(\s*Chủ biên\s*\)\s*$', re.IGNORECASE),
    re.compile(r'^\s*VƯƠNG\s+TRIỀU\s+[A-ZÀ-Ỹ\s]+\s*\(\s*\d{4}\s*-\s*\d{4}\s*\)\s*$', re.IGNORECASE),
    re.compile(r'^\s*\d{1,4}\s*$', re.MULTILINE),
]


def extract_all_pages(pdf_path: str):
    START_PAGE = 9
    END_PAGE   = 769
    results = []

    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(tqdm(pdf.pages, desc=" Đang xử lý trang"), start=1):
            if i < START_PAGE:
                continue

            if i > END_PAGE:
                break

            separator_y = detect_footer_separator(page)

            if separator_y:
                # === CÓ ĐƯỜNG KẼ → Tách rõ ràng main + footer ===
                main_area = page.crop((0, 0, page.width, separator_y - 2))      # trừ margin nhỏ
                foot_area = page.crop((0, separator_y + 2, page.width, page.height))

                raw_main = main_area.extract_text(x_tolerance=3, y_tolerance=3) or ""
                raw_foot = foot_area.extract_text(x_tolerance=3, y_tolerance=3) or ""

                raw_text = raw_main.strip() + "\n---FOOTNOTE_BOUNDARY---\n" + raw_foot.strip()
            else:
                # === Không tìm thấy đường kẻ → lấy toàn bộ ===
                raw_text = page.extract_text(x_tolerance=3, y_tolerance=3) or ""

            if not raw_text.strip():
                continue

            text, fn_dict = clean_page(raw_text)

            results.append({
                "page_num": i,
                "text": text,
                "footnotes": fn_dict
            })

    return results



In [23]:

def main():
    print(f"Sách: {BOOK_NAME}")
    pages = extract_all_pages(PDF_PATH)

    with open("pages_VTT_data1.json", "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)

    full_text, page_fn_map = build_full_text(pages)

    full_text_data = {
        "book_name": BOOK_NAME,
        "full_text": full_text,
        "page_footnotes_map": page_fn_map
    }

    with open("full_VTT_data1.json", "w", encoding="utf-8") as f:
        json.dump(full_text_data, f, ensure_ascii=False, indent=2)


    from google.colab import files

    files.download("pages_VTT_data1.json")
    files.download("full_VTT_data1.json")


if __name__ == "__main__":
    main()

Sách: Vương Triều Trần (1226-1400)


 Đang xử lý trang:  95%|█████████▌| 769/806 [03:28<00:10,  3.68it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# VNSL_TranTrongKim

In [24]:
import re
import json
import pdfplumber
from tqdm import tqdm


PDF_PATH      = "VNSL_TranTrongKim.pdf"
OUTPUT_JSON   = "VNSL_TranTrongKim_chunks.json"
BOOK_NAME     = "Việt Nam Sử Lược"


In [25]:

# Separator ưu tiên tách tại ranh giới đoạn/câu tự nhiên của văn bản sử ký
SEPARATORS = [
    "\n\n",   # Ranh giới đoạn (ưu tiên cao nhất)
    "\n",     # Ranh giới dòng
    ". ",     # Cuối câu
    "! ", "? ", "; ", ", ", " ", "",
]

PAGE_MARKER_FMT = "\n[[[PAGE:{page}]]]\n"

# Làm sách trang
def normalize_whitespace(text: str) -> str:
    text = re.sub(r'[ \t]+\n', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(
        r'(?<![.!?;:])\n(?!\n)',
        ' ',
        text
    )
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def clean_page(raw_text: str) -> tuple[str, dict]:
    if not raw_text:
        return "", {}
    text = re.sub(r'\[\d{1,2}[ab]\]', '', text)
    fn_dict = {}
    text = normalize_whitespace(text)
    return text, fn_dict


def extract_all_pages(pdf_path: str) -> list[dict]:
    results = []
    with pdfplumber.open(pdf_path) as pdf:
        total = len(pdf.pages)
        print(f"Mở PDF: {total} trang")
        for i, page in enumerate(tqdm(pdf.pages, desc="📄 Đọc trang", unit="tr"), start=1):
            raw = page.extract_text(x_tolerance=3, y_tolerance=3)
            if not raw or len(raw.strip()) < 20:
                continue

            text, fn_dict = clean_page(raw)

            if not text or len(text.strip()) < 20:
                continue
            results.append({"page_num": i, "text": text, "footnotes": fn_dict})
    return results


def build_full_text(pages: list[dict]) -> tuple[str, dict]:
    parts = []
    page_fn_map = {}

    for page in pages:
        pn = page["page_num"]
        page_fn_map[pn] = page["footnotes"]
        # Đặt marker trước mỗi trang để track page boundary sau split
        parts.append(PAGE_MARKER_FMT.format(page=pn))
        parts.append(page["text"])

    return "".join(parts), page_fn_map

def clean_page(raw_text: str) -> tuple[str, dict]:
    if not raw_text:
        return "", {}
    text = re.sub(r'[\[\d{1,2}[ab]\]', '', raw_text) # Fix: use raw_text instead of undefined 'text'
    fn_dict = {}
    text = normalize_whitespace(text)
    return text, fn_dict


In [26]:


def main():
    print(f"Sách: {BOOK_NAME}")
    pages = extract_all_pages(PDF_PATH)

    with open("pages_SuLuoc_data.json", "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)

    full_text, page_fn_map = build_full_text(pages)

    full_text_data = {
        "book_name": BOOK_NAME,
        "full_text": full_text,
        "page_footnotes_map": page_fn_map
    }

    with open("full_SuLuoc_data.json", "w", encoding="utf-8") as f:
        json.dump(full_text_data, f, ensure_ascii=False, indent=2)


    from google.colab import files

    files.download("pages_SuLuoc_data.json")
    files.download("full_SuLuoc_data.json")


if __name__ == "__main__":
    main()

Sách: Việt Nam Sử Lược
Mở PDF: 80 trang


📄 Đọc trang: 100%|██████████| 80/80 [00:14<00:00,  5.35tr/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>